In [1]:
import netCDF4 as net
from services.fvc_service               import FvcService
import warnings
import os
import glob
from services.read_ndvi_service         import ReadNdviService
from utilities.utilities_extraction_data import UtilitiesExtractionData
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
warnings.filterwarnings("ignore")
os.environ['HDF5_DISABLE_VERSION_CHECK'] = '2'

In [3]:
month = '01'

In [4]:
files_myd_ = '/home/sagonda/Documentos/git/projects/Prototype/myd21/2016/01/MYD21.A2016015.1150.061.2021341223010.hdf'
print(files_myd_)

/home/sagonda/Documentos/git/projects/Prototype/myd21/2016/01/MYD21.A2016015.1150.061.2021341223010.hdf


In [5]:
files_ = '/media/sagonda/Elements1/2016/01/15/TES_20160115_1150_Day.nc'
print(files_)

/media/sagonda/Elements1/2016/01/15/TES_20160115_1150_Day.nc


In [6]:
data = net.Dataset(files_)
data_myd = net.Dataset(files_myd_) 
# del files, files_myd

In [7]:
e32_original_list = data.variables['UVEG_e32'][:,:] 
e32_myd_list = data_myd.variables['Emis_32'][:,:]

In [8]:
e32_original_float = (e32_original_list * 0.002) + 0.49 
e32_myd_float = (e32_myd_list* 0.002) + 0.49  

In [9]:
# e32_original_float_mask = np.where(e32_original_float > 0.49)
e32_original_float = np.where(e32_original_float == 0.49, np.nan, e32_original_float)

In [10]:
e32_original_float

array([[  nan,   nan,   nan, ..., 0.97 ,   nan,   nan],
       [  nan,   nan,   nan, ...,   nan, 0.974, 0.974],
       [  nan,   nan,   nan, ...,   nan,   nan,   nan],
       ...,
       [0.978, 0.978, 0.98 , ...,   nan,   nan,   nan],
       [0.98 , 0.98 , 0.982, ...,   nan,   nan,   nan],
       [0.982, 0.98 , 0.98 , ...,   nan,   nan,   nan]])

In [11]:
latitud = data.variables['lat'][:,:]/10000#[e32_original_float_mask]/10000 
longitud = data.variables['lon'][:,:]/10000#[e32_original_float_mask]/10000
del data

In [12]:
ndvi_lat, ndvi_lon, ndvi = ReadNdviService.read_ndvi_file('/media/sagonda/6D53615C4A88515D/data/ndvi/2016/', '2016', month)

####################
Read files NDVI.................
NDVI files upload!!
####################


In [13]:
ndvi_ = UtilitiesExtractionData.extract_ndvi(latitud, longitud, ndvi_lat, ndvi_lon, ndvi)

In [14]:
import numpy as np
def FVC(ndvi, e32):
    try:
        fcv = np.empty(shape=(ndvi.shape))
        e32_= np.empty(shape=(ndvi.shape))
        e32_0= np.empty(shape=(ndvi.shape))


        fcv[:] = ((ndvi[:] - 0.3) / np.float32(1 - 0.3))**2
        fcv[:] = np.where(fcv > 1, 1, fcv)
        fcv[:] = np.where(fcv < 0, np.nan, fcv)

        e32_[:] = e32.ravel()[:]
        e32_0[:] = e32.ravel()[:]

        c_veg = np.where(ndvi >= 0.3)
        # c_soil = np.where(self.ndvi < 0.2)
        #condicon mayores de 0.48 y menos de 0.92
        e32_[c_veg] = np.where(e32_[c_veg] < 0.9, 0.974 + 0.015 * fcv[c_veg], e32_[c_veg])
        e32_0[c_veg] = np.where(e32_[c_veg], 0.974 + 0.015 * fcv[c_veg], e32_[c_veg])


        print(e32_.shape)
        return e32_, e32_0

    except OSError as err:
            print("OS error: {0}".format(err))

In [15]:
e32_fvc, e32_fvc_0 = FVC(ndvi_, e32_original_float)

(2748620,)


In [16]:
print(latitud.shape, longitud.shape, e32_fvc.shape, e32_fvc_0.shape, e32_myd_float.shape, ndvi_.shape)

(2030, 1354) (2030, 1354) (2748620,) (2748620,) (2030, 1354) (2748620,)


In [17]:
ndvi_ = np.reshape(ndvi_, newshape=(2030, 1354))
e32_fvc = np.reshape(e32_fvc, newshape=(2030, 1354))
e32_fvc_0 = np.reshape(e32_fvc_0, newshape=(2030, 1354))

In [18]:
print(latitud.shape, longitud.shape, e32_fvc.shape, e32_myd_float.shape, ndvi_.shape)

(2030, 1354) (2030, 1354) (2030, 1354) (2030, 1354) (2030, 1354)


In [19]:
import h5py
# Nombre del archivo HDF
nombre_archivo = '/media/sagonda/6D53615C4A88515D/output/imagen_corregida.h5'

# Crear el archivo HDF y guardar los datos
with h5py.File(nombre_archivo, 'w') as archivo_hdf:
    # Guardar las matrices en el archivo HDF
    archivo_hdf.create_dataset('latitud', data=latitud)
    archivo_hdf.create_dataset('longitud', data=longitud)
    archivo_hdf.create_dataset('E32_UVEG', data=e32_fvc)
    archivo_hdf.create_dataset('E32_UVEG_sin_condicion', data=e32_fvc_0)
    archivo_hdf.create_dataset('E32_MYD', data=e32_myd_float)
    archivo_hdf.create_dataset('NDVI', data=ndvi_)

print(f'Datos guardados en {nombre_archivo}')

Datos guardados en /media/sagonda/6D53615C4A88515D/output/imagen_corregida.h5
